# 第 6 章 · Observation 如何回到上下文

**这一章你会得到什么**：看清命令执行的**输出**是怎么被格式化成一条模型能读的消息，重新进入 `messages` 的；并彻底理解 `tool` 与 `user` 两种 role 的分界。

In [2]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

mini-SWE-agent: 2.4.5


In [5]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 概念：observation 的生成

`format_toolcall_observation_messages` 拿到 `(actions, outputs)` 两个等长列表，
用 `observation_template`（Jinja）把每个 output 渲染成 content，并按是否有 `tool_call_id` 决定 role。

## 实验 1：把一条执行输出渲染成 observation 消息

In [7]:
from minisweagent.models.utils.actions_toolcall import format_toolcall_observation_messages
TEMPLATE = "<returncode>{{output.returncode}}</returncode>\n<output>\n{{output.output}}</output>"
msgs = format_toolcall_observation_messages(
    actions=[{"command": "ls", "tool_call_id": "call_1"}],
    outputs=[{"output": "a.txt\nb.txt", "returncode": 0, "exception_info": ""}],
    observation_template=TEMPLATE,
)
for m in msgs:
    print("role:", m["role"], "| tool_call_id:", m.get("tool_call_id"))
    print(m["content"])

role: tool | tool_call_id: call_1
<returncode>0</returncode>
<output>
a.txt
b.txt</output>


## 实验 2：`tool` vs `user`——同一个函数，两种 role

关键分支：action 里**有** `tool_call_id` → `role="tool"`（模型的工具调用结果）；
**没有** `tool_call_id` → `role="user"`（人类手动敲的命令）。跑一下对比。

In [8]:
with_id = format_toolcall_observation_messages(
    actions=[{"command": "ls", "tool_call_id": "call_1"}],
    outputs=[{"output": "x", "returncode": 0}],
    observation_template=TEMPLATE,
)[0]
without_id = format_toolcall_observation_messages(
    actions=[{"command": "ls"}],  # 没有 tool_call_id
    outputs=[{"output": "x", "returncode": 0}],
    observation_template=TEMPLATE,
)[0]
print("有 tool_call_id ->", with_id["role"], with_id.get("tool_call_id"))
print("无 tool_call_id ->", without_id["role"], without_id.get("tool_call_id"))

有 tool_call_id -> tool call_1
无 tool_call_id -> user None


## 观察点

- `tool_call_id` 必须和 assistant 消息里 `tool_calls[i].id` 配对，API 才认；这是四种 role 协议里最容易被忽略的“暗契约”。
- 人类手敲命令没有 tool call，所以退化成 `user` role——这正是交互式 Agent（第 11 章）的用法。

## 实验 3：outputs 少于 actions 时会发生什么？

源码用一个“未执行占位符”把 outputs 补齐到和 actions 一样长。造一个 2 actions、1 output 的场景看看。

In [ ]:
padded = format_toolcall_observation_messages(
    actions=[{"command": "a", "tool_call_id": "c1"}, {"command": "b", "tool_call_id": "c2"}],
    outputs=[{"output": "done-a", "returncode": 0}],   # 只有一个 output
    observation_template=TEMPLATE,
)
for m in padded:
    print(m["tool_call_id"], "->", repr(m["content"]), "| exception:", m["extra"]["exception_info"])

## 观察点
- 第二个 action 没有对应 output，被补上 `{"returncode": -1, "exception_info": "action was not executed"}`。
- 为什么要补齐？因为 API 要求**每个** tool call 都必须有一条对应的 tool 结果消息，缺一条就会报错。这是 harness 必须替你兜住的细节。

## 动手：给多命令批次配一个“部分失败”的 output
补全下面的 outputs，让第一个命令成功、第二个命令返回码非 0，然后观察两条 observation 的差异。

In [9]:
# TODO: 把 outputs 写成两个 dict：第一个 returncode=0，第二个 returncode=1 且带 exception_info
outputs = [
    {"output": "ok", "returncode": 0},
    {"output": "not ok", "returncode": 1, "exception_info": {"x": "x", "y": "y"}},  # TODO 替换成失败的 output dict
]
# 写完后取消下面注释运行：
for m in format_toolcall_observation_messages(
    actions=[{"command": "a", "tool_call_id": "c1"}, {"command": "b", "tool_call_id": "c2"}],
    outputs=outputs, observation_template=TEMPLATE):
    print(m["extra"]["returncode"], m["content"])

0 <returncode>0</returncode>
<output>
ok</output>
1 <returncode>1</returncode>
<output>
not ok</output>


In [4]:
show_source("src/minisweagent/models/utils/actions_toolcall.py", 79, 113)

 79  def format_toolcall_observation_messages(
 80      *,
 81      actions: list[dict],
 82      outputs: list[dict],
 83      observation_template: str,
 84      template_vars: dict | None = None,
 85      multimodal_regex: str = "",
 86  ) -> list[dict]:
 87      """Format execution outputs into tool result messages."""
 88      not_executed = {"output": "", "returncode": -1, "exception_info": "action was not executed"}
 89      padded_outputs = outputs + [not_executed] * (len(actions) - len(outputs))
 90      results = []
 91      for action, output in zip(actions, padded_outputs):
 92          content = Template(observation_template, undefined=StrictUndefined).render(
 93              output=output, **(template_vars or {})
 94          )
 95          msg = {
 96              "content": content,
 97              "extra": {
 98                  "raw_output": output.get("output", ""),
 99                  "returncode": output.get("returncode"),
100                  "timestamp": time.

## 闭卷检查
1. observation 的 role 由什么决定？
2. `tool_call_id` 为什么必须配对？
3. outputs 比 actions 少时，harness 如何兜底、为什么必须兜底？